In [ ]:
<a href="https://colab.research.google.com/github/blhprasanna99/speech_emotion_detection/blob/master/Speechemotion_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SPEECH EMOTION DETECTION USING MLP

This notebook is customized for your own dataset with three classes: **satisfied**, **unsatisfied**, and **average**.

In [ ]:
# If running locally and you don't have these, uncomment and run:
!pip install soundfile librosa scikit-learn matplotlib seaborn

In [ ]:
import soundfile
import numpy as np
import librosa
import glob
import os
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Custom emotion mapping
custom_emotions_map = {
    'satisfied': 0,
    'unsatisfied': 1,
    'average': 2
}
custom_emotion_names = ['satisfied', 'unsatisfied', 'average']

In [ ]:
def extract_feature(file_name, **kwargs):
    """
    Extract feature from audio file `file_name`
    Features supported:
        - MFCC (mfcc)
        - Chroma (chroma)
        - MEL Spectrogram Frequency (mel)
        - Contrast (contrast)
        - Tonnetz (tonnetz)
    e.g:
    `features = extract_feature(path, mel=True, mfcc=True)`
    """
    mfcc = kwargs.get("mfcc")
    chroma = kwargs.get("chroma")
    mel = kwargs.get("mel")
    contrast = kwargs.get("contrast")
    tonnetz = kwargs.get("tonnetz")
    with soundfile.SoundFile(file_name) as sound_file:
        X = sound_file.read(dtype="float32")
        sample_rate = sound_file.samplerate
        if chroma or contrast:
            stft = np.abs(librosa.stft(X))
        result = np.array([])
        if mfcc:
            mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
            result = np.hstack((result, mfccs))
        if chroma:
            chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T,axis=0)
            result = np.hstack((result, chroma))
        if mel:
            mel = np.mean(librosa.feature.melspectrogram(X, sr=sample_rate).T,axis=0)
            result = np.hstack((result, mel))
        if contrast:
            contrast = np.mean(librosa.feature.spectral_contrast(S=stft, sr=sample_rate).T,axis=0)
            result = np.hstack((result, contrast))
        if tonnetz:
            tonnetz = np.mean(librosa.feature.tonnetz(y=librosa.effects.harmonic(X), sr=sample_rate).T,axis=0)
            result = np.hstack((result, tonnetz))
    return result

In [ ]:
def load_custom_data(data_root_path, test_size=0.25):
    x, y = [], []
    for emotion_folder_name in custom_emotions_map.keys():
        current_folder_path = os.path.join(data_root_path, emotion_folder_name)
        for file_path in glob.glob(os.path.join(current_folder_path, '*.wav')):
            label_integer = custom_emotions_map[emotion_folder_name]
            feature = extract_feature(file_path, mfcc=True, chroma=True, mel=True)
            x.append(feature)
            y.append(label_integer)
    X = np.array(x)
    Y = np.array(y)
    return train_test_split(X, Y, test_size=test_size, random_state=9, stratify=Y)

In [ ]:
data_root_path = 'my_audio'  # Make sure this matches your folder name
X_train, X_test, y_train, y_test = load_custom_data(data_root_path, test_size=0.25)

print("[+] Number of training samples:", X_train.shape[0])
print("[+] Number of testing samples:", X_test.shape[0])
print("[+] Number of features:", X_train.shape[1])

In [ ]:
model_params = {
    'alpha': 0.01,
    'batch_size': 256,
    'epsilon': 1e-08, 
    'hidden_layer_sizes': (300,), 
    'learning_rate': 'adaptive', 
    'max_iter': 500, 
}
model = MLPClassifier(**model_params)

print("[*] Training the model...")
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_true=y_test, y_pred=y_pred)
print("Accuracy: {:.2f}%".format(accuracy*100))

print(classification_report(y_test, y_pred, target_names=custom_emotion_names))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=custom_emotion_names, yticklabels=custom_emotion_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Change the path below to any .wav file you want to test
test_file_path = "my_audio/satisfied/your_test_file.wav"  # Change as needed

test_feature = extract_feature(test_file_path, mfcc=True, chroma=True, mel=True)
test_feature_reshaped = np.expand_dims(test_feature, axis=0)
prediction = model.predict(test_feature_reshaped)[0]
predicted_emotion = custom_emotion_names[prediction]
print(f"The predicted emotion for '{os.path.basename(test_file_path)}' is: {predicted_emotion}")

In [ ]:
import pickle
if not os.path.isdir("result"):
    os.mkdir("result")
pickle.dump(model, open("result/mlp_classifier.model", "wb"))